In [ ]:
import torch
import torch.nn as nn

class MLPEncoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=16, hidden_dims=None, dropout_rate=0.2, use_layernorm=True):
        """
        更适合大规模表格数据的MLP Encoder。
        - hidden_dims: 自动根据输入维度设置，或自定义
        - LayerNorm: 对小batch更稳定，适合联邦学习
        - Kaiming初始化：提升训练稳定性
        - 输出L2归一化，便于下游融合
        """
        super().__init__()
        if hidden_dims is None:
            h1 = max(128, min(512, input_dim * 2))
            h2 = max(64, min(256, input_dim))
            hidden_dims = [h1, h2]
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU(inplace=True))
            if use_layernorm:
                layers.append(nn.LayerNorm(h))
            else:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(dropout_rate))
            in_dim = h
        layers.append(nn.Linear(in_dim, embedding_dim))
        self.encoder = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, a=0.0, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        x = x.float()
        emb = self.encoder(x)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        return emb

import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# 1. 加载数据
df = pd.read_csv('tabular_dataset/diabetes_012_ready_to_model.csv', index_col=0)
feature_cols = [c for c in df.columns if not c.startswith('Diabetes_class_')]
X_df = df[feature_cols].astype(np.float32)
input_dim = X_df.shape[1]

# 2. 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

# 3. DataLoader分批处理（适合大数据量）
batch_size = 4096
tensor_ds = TensorDataset(torch.from_numpy(X_scaled.astype(np.float32)))
dl = DataLoader(tensor_ds, batch_size=batch_size, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLPEncoder(input_dim=input_dim, embedding_dim=16, hidden_dims=[128, 64, 32], dropout_rate=0.2).to(device)
model.eval()

embeddings_list = []
with torch.no_grad():
    for (xb,) in dl:
        xb = xb.to(device)
        emb = model(xb)
        embeddings_list.append(emb.cpu())
embeddings = torch.cat(embeddings_list, dim=0)
print(f"原始数据维度: {X_df.shape}")
print(f"嵌入向量维度: {embeddings.shape}")

# 可选：保存嵌入向量
np.save("tabular_embeddings.npy", embeddings.numpy())

In [ ]:

# use "tabular_dataset/diabetes_012_ready_to_model.csv" as the input .csv file
# 